In [ ]:
# Cài đặt thư viện quan trọng - Chú ý phiên bản pyannote và torch để không bị conflict
!pip install pyannote.audio==4.0.0
!pip install torch==2.8.0 --index-url https://download.pytorch.org/whl/cu126
!pip install torchaudio==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu126
!pip install torchcodec==0.7
!pip install -q 'datasets>=2.14' soundfile pyannote.metrics

print("\n✅ Cài đặt xong!")
print("🔄 RESTART RUNTIME: Runtime → Restart session")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import torch
import numpy as np
from google.colab import drive

# Mount Drive để tí nữa lấy checkpoint
drive.mount('/content/drive')

# PyTorch patch để tránh lỗi weights_only khi load checkpoint cũ
_orig_load = torch.load
def _patched_load(f, *args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_load(f, *args, **kwargs)
torch.load = _patched_load
print("✓ PyTorch load function patched")

# NumPy patch cho mấy bản cũ hay lỗi thuộc tính gán nhãn
if not hasattr(np, 'NaN'):
    np.NaN = np.nan
if not hasattr(np, 'Inf'):
    np.Inf = np.inf

# Kiểm tra xem Colab có cấp GPU ngon không (T4, A100...)
assert torch.cuda.is_available(), "❌ Không có GPU! Nhớ đổi Runtime sang GPU trước khi chạy."
device = torch.device('cuda')
print(f'✓ PyTorch {torch.__version__}')
print(f'✓ GPU đang dùng: {torch.cuda.get_device_name(0)}')

In [ ]:
# Token HuggingFace dùng để kéo model pyannote gated và dataset voxconverse
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✅ Đọc token từ Colab Secrets thành công")
except Exception:
    # Nếu chạy local hoặc chưa set secret thì điền tay vào đây nhé
    HF_TOKEN = "YOUR_HF_TOKEN"
    print("⚠️ Dùng token điền tay")

assert HF_TOKEN and not HF_TOKEN.startswith("hf_PASTE"), "❌ Chưa điền HuggingFace token!"
print(f"✅ Token hợp lệ: {HF_TOKEN[:8]}...")

In [ ]:
# ============================================================
# ⚙️ CẤU HÌNH ĐÁNH GIÁ (Sửa các tham số ở đây nếu cần thay đổi data)
# ============================================================
TOTAL_PARTS         = 3
SAMPLES_PER_PART    = 5   # số mẫu test cho mỗi part

BASE_DRIVE_DIR = '/content/drive/MyDrive/Diarization_Continual_Learning'

# Thư mục làm việc tạm thời trên local Colab — sẽ tải audio test về đây cho nhanh
WORK_DIR      = '/content/voxconverse_eval'
AUDIO_DIR     = os.path.join(WORK_DIR, 'audio')
RTTM_DIR      = os.path.join(WORK_DIR, 'rttm')
UEM_DIR       = os.path.join(WORK_DIR, 'uem')
OUTPUT_DIR    = '/content/output_eval'

for d in [AUDIO_DIR, RTTM_DIR, UEM_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✓ TOTAL_PARTS      : {TOTAL_PARTS}")
print(f"✓ SAMPLES_PER_PART : {SAMPLES_PER_PART}")
print(f"✓ BASE_DRIVE_DIR   : {BASE_DRIVE_DIR}")
print(f"✓ Thư mục làm việc : {WORK_DIR}")

In [ ]:
import soundfile as sf
from datasets import load_dataset

def make_rttm(uri, starts, ends, speakers):
    lines = []
    for s, e, spk in zip(starts, ends, speakers):
        dur = round(float(e) - float(s), 3)
        if dur <= 0:
            continue
        lines.append(
            f"SPEAKER {uri} 1 {float(s):.3f} {dur:.3f} <NA> <NA> {spk} <NA> <NA>"
        )
    return "\n".join(lines) + "\n" if lines else ""

def make_uem(uri, duration):
    return f"{uri} 1 0.000 {duration:.3f}\n"

total_test_samples = TOTAL_PARTS * SAMPLES_PER_PART
print(f"📥 Streaming {total_test_samples} mẫu từ VoxConverse test split...")

ds_test = load_dataset(
    "diarizers-community/voxconverse",
    split="test",
    token=HF_TOKEN,
    streaming=True)
all_test_samples = list(ds_test.take(total_test_samples))
print(f"✅ Đã tải xong {len(all_test_samples)} mẫu")

# Lưu audio + RTTM + UEM ra disk, phân chia rạch ròi theo part
all_uris = []   # danh sách uri tracking tuyến tính
print("\n⚙️ Đang lưu audio / RTTM / UEM...")

for global_idx, sample in enumerate(all_test_samples):
    part      = global_idx // SAMPLES_PER_PART + 1
    local_idx = global_idx  % SAMPLES_PER_PART
    uri       = f"p{part}_{local_idx:03d}"
    all_uris.append(uri)

    # Ghi file Audio (.wav)
    audio = sample['audio']
    arr   = np.array(audio['array'], dtype=np.float32)
    sr    = audio['sampling_rate']
    sf.write(os.path.join(AUDIO_DIR, f"{uri}.wav"), arr, sr)
    duration = len(arr) / sr

    # Ghi file Ground-truth RTTM
    with open(os.path.join(RTTM_DIR, f"{uri}.rttm"), 'w') as f:
        f.write(make_rttm(uri, sample['timestamps_start'],
                          sample['timestamps_end'], sample['speakers']))

    # Ghi file UEM (vùng tính toán hợp lệ)
    with open(os.path.join(UEM_DIR, f"{uri}.uem"), 'w') as f:
        f.write(make_uem(uri, duration))

    n_spk = len(set(sample['speakers']))
    print(f"  [{global_idx+1:02d}/{total_test_samples}] {uri} | dur={duration:.1f}s | speakers={n_spk}")

# Group lại theo từng part để tiện loop chạy ma trận kết quả ở cell sau
uris_per_part = {}
for p in range(1, TOTAL_PARTS + 1):
    uris_per_part[p] = [u for u in all_uris if u.startswith(f"p{p}_")]

print("\n✅ Hoàn tất lưu dữ liệu:")
for p, uris in uris_per_part.items():
    print(f"  Part {p}: {uris}")

In [ ]:
from pyannote.audio import Model, Inference, Pipeline

def load_seg_model_from_ckpt(ckpt_path, hf_token, device):
    """Load segmentation model từ Lightning .ckpt: khởi tạo HF arch rồi override backbone."""
    model = Model.from_pretrained('pyannote/segmentation-3.0', token=hf_token)
    ckpt = torch.load(ckpt_path, map_location=device)
    assert 'state_dict' in ckpt, f"Không tìm thấy 'state_dict' trong {ckpt_path}"

    # Loại bỏ layer classifier vì size mismatch do số speakers không cố định qua các part
    state_dict = {k: v for k, v in ckpt['state_dict'].items()
                  if not k.startswith('classifier')}

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    backbone_missing = [k for k in missing if not k.startswith('classifier')]

    assert not backbone_missing, f"❌ Thiếu backbone keys quan trọng: {backbone_missing}"
    assert not unexpected,       f"❌ Trọng số lạ không khớp kiến trúc: {unexpected}"

    model.to(device)
    model.eval()
    return model, ckpt.get('epoch', 'N/A')

def find_best_ckpt(part):
    """Quét thư mục tìm checkpoint ngon nhất (best-model...) hoặc bí quá thì dùng last.ckpt"""
    ckpt_dir = os.path.join(BASE_DRIVE_DIR, f'checkpoints_part_{part}')
    if not os.path.isdir(ckpt_dir):
        return None
    best = sorted([f for f in os.listdir(ckpt_dir) if f.startswith('best-model')])
    if best:
        return os.path.join(ckpt_dir, best[-1])
    last = os.path.join(ckpt_dir, 'last.ckpt')
    return last if os.path.exists(last) else None

# ── Thực hiện load hàng loạt ──
models = {} # 0: baseline, 1..3: checkpoints sau từng chặng train

print("Loading Model 0 (pretrained baseline from HuggingFace)...")
models[0] = Model.from_pretrained('pyannote/segmentation-3.0', token=HF_TOKEN)
models[0].to(device)
models[0].eval()
print("  ✓ Model 0 baseline ready")

for part in range(1, TOTAL_PARTS + 1):
    ckpt_path = find_best_ckpt(part)
    if ckpt_path is None:
        print(f"  ⚠️ Không tìm thấy checkpoint cho Part {part} — Bỏ qua model này.")
        continue
    print(f"\nLoading Model {part} (Trọng số sau khi học xong Part {part})...")
    print(f"  Checkpoint path: {os.path.basename(ckpt_path)}")
    models[part], epoch = load_seg_model_from_ckpt(ckpt_path, HF_TOKEN, device)
    print(f"  ✓ Model {part} ready (được lưu tại epoch {epoch})")

print(f"\n✅ Tổng số model đã nạp thành công: {len(models)}")
print(f"   Keys hiện có: {sorted(models.keys())} (0=gốc, 1..N=sau khi học continual learning)")

In [ ]:
print("Đang tải pipeline phân tách người nói mặc định...")
base_pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-community-1',
    token=HF_TOKEN)
base_pipeline.to(device)
print("✓ Base pipeline loaded")

def make_pipeline_with_model(seg_model, base_pipeline, device):
    """Tráo đổi mô hình segmentation cũ bằng mô hình mới tinh chỉnh mà giữ nguyên cấu hình pipeline"""
    from copy import deepcopy
    new_pipeline = deepcopy(base_pipeline)
    new_pipeline._segmentation = Inference(
        seg_model,
        duration=base_pipeline._segmentation.duration,
        step=base_pipeline._segmentation.step,
        skip_aggregation=base_pipeline._segmentation.skip_aggregation,
        batch_size=base_pipeline._segmentation.batch_size,
    )
    new_pipeline.to(device)
    assert new_pipeline._segmentation.model is seg_model, "❌ Lỗi swap mô hình đè nhau!"
    return new_pipeline

print("✓ Hàm swap mô hình phân đoạn đã sẵn sàng sử dụng!")

In [ ]:
from pyannote.metrics.diarization import DiarizationErrorRate, JaccardErrorRate
from pyannote.core import Annotation, Segment
from pyannote.audio import Audio

audio_loader = Audio(mono='downmix')

def load_rttm_annotation(rttm_path):
    """Hàm parse thủ công file RTTM text thành object Annotation cấu trúc dữ liệu của Pyannote"""
    ann = Annotation()
    with open(rttm_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith(';'):
                continue
            parts = line.split()
            start    = float(parts[3])
            duration = float(parts[4])
            speaker  = parts[7]
            ann[Segment(start, start + duration)] = speaker
    return ann

def evaluate_on_uris(pipeline, uri_list, audio_dir, rttm_dir):
    """Hàm quét một lượt qua list file, suy luận (infer) và đối chiếu tính DER/JER tổng quát"""
    der_metric = DiarizationErrorRate()
    jer_metric = JaccardErrorRate()
    per_file   = []

    for uri in uri_list:
        audio_path = os.path.join(audio_dir, f"{uri}.wav")
        rttm_path  = os.path.join(rttm_dir,  f"{uri}.rttm")

        reference  = load_rttm_annotation(rttm_path)
        output     = pipeline({'audio': audio_path})

        # DiarizeOutput.speaker_diarization là Annotation object
        hypothesis = output.speaker_diarization if isinstance(output, Annotation) is False else output

        der = der_metric(reference, hypothesis)
        jer = jer_metric(reference, hypothesis)

        per_file.append({'uri': uri, 'DER': der * 100, 'JER': jer * 100})

    total_der = abs(der_metric) * 100
    total_jer = abs(jer_metric) * 100
    return {'DER': total_der, 'JER': total_jer, 'per_file': per_file}

print("✓ Đã dựng xong bộ helper tính metric DER/JER.")

In [ ]:
import time

# Khởi tạo dict chứa ma trận kết quả: results[model_key][part] = {DER, JER, per_file}
results = {}
model_keys = sorted(models.keys())  # [0, 1, 2, 3]

# Đoạn này chạy hơi lâu xíu tùy thuộc vào độ dài audio và số file test
for model_key in model_keys:
    label = "baseline (HF pretrained)" if model_key == 0 else f"after Part {model_key}"
    print(f"\n{'='*60}")
    print(f"  🔥 Đang đánh giá Model {model_key} — {label}")
    print(f"{'='*60}")

    # Build pipeline kết hợp với model phân đoạn tương ứng
    pipeline_i = make_pipeline_with_model(models[model_key], base_pipeline, device)
    results[model_key] = {}

    for part in range(1, TOTAL_PARTS + 1):
        uris = uris_per_part[part]
        print(f"  → Đánh giá trên Test Part {part} ({len(uris)} files)...", end=" ", flush=True)

        t0 = time.time()
        res = evaluate_on_uris(pipeline_i, uris, AUDIO_DIR, RTTM_DIR)
        elapsed = time.time() - t0

        results[model_key][part] = res
        print(f"Done! DER={res['DER']:.2f}% | JER={res['JER']:.2f}% ({elapsed:.1f}s)")

print("\n✅ Quá trình test chéo hoàn tất! Chạy tiếp cell dưới để xem ma trận trực quan.")

In [ ]:
# ============================================================
# IN MA TRẬN KẾT QUẢ ĐỂ PHÂN TÍCH (CATASTROPHIC FORGETTING)
# ============================================================
model_keys = sorted(results.keys())
part_keys  = list(range(1, TOTAL_PARTS + 1))
col_w    = 22
label_w  = 26

def model_label(k):
    return "Model 0 (baseline)" if k == 0 else f"Model {k} (after Part {k})"

def part_label(p):
    return f"Test Part {p} ({SAMPLES_PER_PART} files)"

# ── IN MA TRẬN DER ──────────────────────────────────────────────────────
print("\n" + "="*80)
print("  📊 DIARIZATION ERROR RATE (DER) — Ma trận đánh giá chéo")
print("="*80)
header = f"{'':>{label_w}}" + "".join(f"{part_label(p):^{col_w}}" for p in part_keys)
print(header)
print("-" * (label_w + col_w * len(part_keys)))

for mk in model_keys:
    row = f"{model_label(mk):>{label_w}}"
    for p in part_keys:
        val = results[mk][p]['DER']
        marker = " ◀" if mk == p else "  " # Đánh dấu đường chéo chính (vừa học xong test luôn)
        row += f"{val:>8.2f}%{marker:>{'12' if marker.strip() else '12'}}"
    print(row)
print("-" * (label_w + col_w * len(part_keys)))
print("💡 Ghi chú: ◀ là đường chéo chính. Nếu các hàng dưới có chỉ số phần cũ tăng mạnh -> Model bị quên kiến thức cũ.")

# ── IN MA TRẬN JER ──────────────────────────────────────────────────────
print("\n" + "="*80)
print("  📊 JACCARD ERROR RATE (JER) — Ma trận đánh giá chéo")
print("="*80)
print(header)
print("-" * (label_w + col_w * len(part_keys)))

for mk in model_keys:
    row = f"{model_label(mk):>{label_w}}"
    for p in part_keys:
        val = results[mk][p]['JER']
        marker = " ◀" if mk == p else "  "
        row += f"{val:>8.2f}%{marker:>{'12' if marker.strip() else '12'}}"
    print(row)
print("-" * (label_w + col_w * len(part_keys)))

In [ ]:
# =====================================================================
# NHÁP THỬ: Xem chi tiết từng file của cặp Model x Part để tìm outlier gây tạ
# =====================================================================
SHOW_MODEL = 1   # ← Đổi key model muốn check sâu (0, 1, 2, 3...)
SHOW_PART  = 1   # ← Đổi số part muốn soi chi tiết

print(f"Chi tiết hiệu năng cụ thể: Model {SHOW_MODEL} × Test Part {SHOW_PART}")
print(f"{'File URI':>12}  {'DER (%)':>8}  {'JER (%)':>8}")
print("-" * 38)

for row in results[SHOW_MODEL][SHOW_PART]['per_file']:
    print(f"{row['uri']:>12}  {row['DER']:>7.2f}%  {row['JER']:>7.2f}%")

In [ ]:
import csv

csv_path = os.path.join(BASE_DRIVE_DIR, 'evaluation_matrix_results.csv')

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    # Header cột dễ xử lý bằng pandas hoặc vẽ chart matplotlib
    writer.writerow(
        ['model'] +
        [f'DER_part{p}' for p in part_keys] +
        [f'JER_part{p}' for p in part_keys]
    )
    for mk in model_keys:
        row = [model_label(mk)]
        row += [f"{results[mk][p]['DER']:.4f}" for p in part_keys]
        row += [f"{results[mk][p]['JER']:.4f}" for p in part_keys]
        writer.writerow(row)

print(f"✅ Kết quả ma trận đã export an toàn ra Drive: {csv_path}")